In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

In [ ]:
import glob
import os
import time

import numpy as np
import pandas as pd
import torch
import easyocr
import onnxruntime as ort

# 스레드 수 고정
torch.set_num_threads(4)

# date_parser.py는 이 노트북과 같은 저장소 루트에 있어야 함
from date_parser import extract_expiry_fields

# [참가자 구현 영역]
image_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.*")))
image_ids = [os.path.splitext(os.path.basename(p))[0] for p in image_files]
print(f"입력 이미지 {len(image_files)}장")

# EasyOCR
reader = easyocr.Reader(
    ['ko', 'en'],
    gpu=False,
    model_storage_directory='./weights',
    download_enabled=False,
    verbose=False,
)

# YOLO(2클래스: EXP=소비기한, MFG=제조일자) — 팀이 직접 라벨링한 450장으로
# yolov8n을 파인튜닝한 뒤 ONNX로 내보낸 것. ultralytics/최신 torch 없이
# onnxruntime만으로 추론해서 EasyOCR 환경(torch 2.2.2)과 충돌이 안 나게 했음.
# 실측 검증(val 45장): mAP50 0.924, EXP 재현율 0.774 — 소비기한 영역을
# 못 찾거나 신뢰도가 낮으면 아래 루프에서 기존 전체이미지 리사이즈 방식으로
# 자동 폴백한다(정확도는 손해 안 보고, 잘 맞는 케이스만 속도 이득을 봄).
YOLO_ONNX_PATH = "./weights/yolo_exp_mfg.onnx"
YOLO_IMGSZ = 640
YOLO_CONF_THRESH = 0.25
YOLO_CROP_MARGIN = 0.15  # 박스 네 변에 각각 15%씩 여유(글자 잘림 방지)

try:
    _yolo_session = ort.InferenceSession(YOLO_ONNX_PATH, providers=["CPUExecutionProvider"])
except Exception as e:
    print(f"[WARN] YOLO ONNX 로드 실패, 전체 리사이즈 방식만 사용: {e}")
    _yolo_session = None

# 리사이즈 긴 변 기준(px) — YOLO가 크롭 못 했을 때의 폴백 경로에서만 씀
RESIZE_LONG_EDGE = 900

In [ ]:
from PIL import Image, ImageOps


def _resize_pil(im, long_edge):
    """긴 변이 long_edge보다 크면 비율 유지해서 줄인다."""
    w, h = im.size
    if max(w, h) <= long_edge:
        return im
    scale = long_edge / max(w, h)
    return im.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.LANCZOS)


def _letterbox(im, size=YOLO_IMGSZ, color=(114, 114, 114)):
    """비율 유지하며 정사각형(size x size)으로 패딩 — YOLO 학습 때 쓴 전처리와 동일."""
    w, h = im.size
    scale = min(size / w, size / h)
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    resized = im.resize((nw, nh), Image.BILINEAR)
    canvas = Image.new("RGB", (size, size), color)
    dw, dh = (size - nw) // 2, (size - nh) // 2
    canvas.paste(resized, (dw, dh))
    return canvas, scale, dw, dh


def _detect_exp_crop(im):
    """EXIF 보정된 원본 PIL 이미지에서 EXP(소비기한) 영역을 찾아 크롭해서 반환.
    YOLO 모델이 없거나, 신뢰도가 기준 미만이거나, 박스가 비정상적으로 작으면
    None — 호출부에서 기존 전체이미지 방식으로 폴백한다."""
    if _yolo_session is None:
        return None
    canvas, scale, dw, dh = _letterbox(im)
    arr = np.asarray(canvas, dtype=np.float32) / 255.0
    arr = arr.transpose(2, 0, 1)[None, ...]
    outputs = _yolo_session.run(None, {"images": arr})[0]  # (1, 6, 8400)
    preds = outputs[0].T  # (8400, 6): cx,cy,w,h,score_EXP,score_MFG
    exp_scores = preds[:, 4]
    best_idx = int(np.argmax(exp_scores))
    best_score = float(exp_scores[best_idx])
    if best_score < YOLO_CONF_THRESH:
        return None

    cx, cy, bw, bh = preds[best_idx, :4]
    x1, y1 = (cx - bw / 2 - dw) / scale, (cy - bh / 2 - dh) / scale
    x2, y2 = (cx + bw / 2 - dw) / scale, (cy + bh / 2 - dh) / scale

    w, h = im.size
    mw, mh = (x2 - x1) * YOLO_CROP_MARGIN, (y2 - y1) * YOLO_CROP_MARGIN
    x1, x2 = max(0, x1 - mw), min(w, x2 + mw)
    y1, y2 = max(0, y1 - mh), min(h, y2 + mh)
    if x2 - x1 < 10 or y2 - y1 < 10:  # 비정상적으로 작은 박스는 오탐으로 보고 폴백
        return None
    return im.crop((int(x1), int(y1), int(x2), int(y2)))


def _ocr_pil(im):
    """PIL 이미지를 파일로 저장하지 않고 바로 EasyOCR에 넘긴다.
    기존 코드가 cv2.imread(BGR) 경로를 썼던 관례에 맞춰 채널 순서를 뒤집는다."""
    arr = np.array(im.convert("RGB"))[:, :, ::-1]
    return " ".join(reader.readtext(arr, detail=0))


results = []
n_cropped = 0
t_start = time.time()
for img_id, path in zip(image_ids, image_files):
    try:
        with Image.open(path) as im_raw:
            # 스마트폰 사진의 EXIF 회전 태그를 실제 픽셀에 반영 — YOLO 학습
            # 데이터도 이렇게 보정한 좌표계 기준이라 이걸 안 하면 크롭 위치가
            # 어긋난다(실측: 라벨링 450장 중 38장이 회전 태그 있었음).
            im = ImageOps.exif_transpose(im_raw).convert("RGB")

        crop = _detect_exp_crop(im)
        crop_text = _ocr_pil(crop).strip() if crop is not None else ""
        if crop_text:
            # 이 텍스트는 YOLO가 EXP(소비기한)로 분류한 영역에서만 나온 것이라
            # 이미 "무슨 종류의 날짜인지" 알고 있음. date_parser.py의 키워드
            # 기반 판별/탐색 로직(부터·까지 우선순위, 키워드 근처(±30~40자)
            # 로만 찾는 안전장치 등)이 정상 작동하도록 그 정보를 명시해준다 —
            # 크롭 자체엔 "소비기한"이라는 글자가 안 찍혀있는 경우가 많아서
            # (라벨이 숫자만 딱 잘라 크롭하니까) 이걸 안 붙이면 그런 로직들이
            # 아예 발동을 안 함(실측: 000009.jpg, "2026 10 29" 공백구분).
            text = "소비기한 " + crop_text
            n_cropped += 1
        else:
            # YOLO가 못 찾았거나(crop is None), 크롭한 영역에서 글자가 하나도
            # 안 읽힌 경우 — 기존 전체이미지 리사이즈 방식으로 폴백
            text = _ocr_pil(_resize_pil(im, RESIZE_LONG_EDGE))
        parsed = extract_expiry_fields(text)
    except Exception as e:
        # 이미지 한 장이 깨져도 전체 실행이 죽지 않고 NONE 처리 후 계속 진행
        print(f"[WARN] {img_id} 처리 중 오류, NONE 처리: {e}")
        parsed = {"year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"}

    results.append({
        "image_id": img_id,
        "year": parsed["year"],
        "month": parsed["month"],
        "day": parsed["day"],
        "final_date": parsed["final_date"],
    })

elapsed = time.time() - t_start
n = max(1, len(results))
print(f"총 {len(results)}장 처리 완료, {elapsed:.1f}초 (장당 평균 {elapsed/n:.2f}초)")
print(f"YOLO 크롭 성공: {n_cropped}장 / 전체이미지 폴백: {len(results) - n_cropped}장")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")